# ISI022 - Sistemas de Informação e Tecnologias Emergentes
### Módulo 3 · Livros-Razão Distribuídos (DLT), Blockchain e Máquinas Virtuais Turing-Completas
**Aula 01 · Fundamentos Criptográficos, Hashes e Arquitetura de Cadeia de Blocos**

* **Docente:** Prof. Me. André Cassulino Araújo Souza (`andre.souza@cps.sp.gov.br`)  
* **Duração Total da Aula:** 1 hora e 40 minutos  
  * **Parte Teórico-Conceitual:** ~60 minutos  
  * **Laboratório Prático Dirigido (Hands-on):** ~40 minutos no simulador visual [blockchaindemo.io](https://blockchaindemo.io/)  
* **Pré-requisitos Consolidados:** Módulos 1 e 2 (Teoria Geral de Sistemas, Pirâmide DIKW, Sistemas Integrados ERP/CRM, Plataformas Digitais e APIs REST).

---

### Objetivos de Aprendizagem da Aula:
1. **Compreender** as vulnerabilidades do modelo tradicional de confiança centralizada (*SPOF*, assimetria de informação) e a formulação do Problema dos Generais Bizantinos.
2. **Dominar** as bases matemáticas e operacionais da Criptografia Assimétrica (chaves pública/privada) e das Funções Hash Criptográficas (SHA-256).
3. **Analisar** o papel estrutural das Árvores de Merkle (*Merkle Trees*) na verificação escalável de transações em tempo logarítmico $\mathcal{O}(\log n)$.
4. **Desmistificar o Conceito de Mineração:** o que é realmente minerar em Engenharia de Software (empacotamento de mempool, validação de assinaturas e busca pelo *Nonce* na Prova de Trabalho).
5. **Compreender a Imutabilidade Coletiva e a Segurança de Ativos:** entender por que a adulteração ou invalidação de um bloco anterior por um invasor **NÃO faz você perder o seu Bitcoin** ou seus ativos na cadeia canônica.
6. **Experimentar e Validar** na prática a quebra em cascata de integridade, a mineração e o consenso distribuído peer-to-peer utilizando o ambiente interativo **https://blockchaindemo.io/**.


---
# 1. O Paradigma da Confiança em Sistemas de Informação

## 1.1 A Fragilidade dos Modelos Centralizados (Web 2.0)
Historicamente, os Sistemas de Informação corporativos apoiaram-se em arquiteturas **cliente-servidor centralizadas**, mediadas por bancos de dados relacionais (RDBMS como Oracle, PostgreSQL e SQL Server). Nesses arranjos, a confiança decorre inteiramente da autoridade de uma entidade guardiã central (instituição financeira, cartório, órgão governamental ou provedor de nuvem).

Embora operem com alta velocidade de processamento transacional (OLTP), arquiteturas centralizadas impõem gargalos estruturais severos:

* **Ponto Único de Falha (*Single Point of Failure - SPOF*):** Caso o nó coordenador central sofra uma indisponibilidade severa (ataque cibernético, corrupção de hardware ou pane de datacenter), toda a cadeia operacional é interrompida.
* **Assimetria Informacional e Risco de Custódia:** Os usuários finais são forçados a confiar cegamente na probidade do administrador do banco de dados, que detém privilégios irrestritos para alterar registros retroativamente (*UPDATE* ou *DELETE* sem rastreabilidade pública).
* **Custos Elevados de Conciliação Contábil:** Em operações interorganizacionais (ex: comércio exterior, compensação bancária e logística internacional), cada empresa mantém seu próprio livro-razão privado, exigindo dias de auditoria e reconciliação manual para sincronizar divergências.

| Dimensão de Análise | Sistemas Centralizados | Sistemas Distribuídos Tradicionais | Redes DLT / Blockchain |
| :--- | :--- | :--- | :--- |
| **Topologia de Nós** | Servidor mestre central com clientes subordinados. | Nós coordenados por balanceadores centrais. | Rede Peer-to-Peer (P2P) descentralizada e simétrica. |
| **Fonte de Confiança** | Autoridade institucional / Contratos legais. | Governança corporativa única / Nuvem. | **Algoritmos criptográficos e consenso de rede**. |
| **Ponto Único de Falha** | Elevado (queda do mestre derruba o sistema). | Moderado (redundância gerenciada centralmente). | **Nulo** (resiliência a falhas bizantinas de nós). |
| **Custódia do Histórico** | Mutável (sujeito a *UPDATE* e *DELETE*). | Mutável conforme privilégios administrativos. | **Imutável** (*Append-only* via encadeamento criptográfico). |

---

## 1.2 O Problema dos Generais Bizantinos
Formalizado por Leslie Lamport, Robert Shostak e Marshall Pease (1982), o **Problema dos Generais Bizantinos** sintetiza o desafio computacional de alcançar consenso e sincronização em uma rede distribuída onde parte dos participantes pode ser desonesta, estar incomunicável ou enviar dados contraditórios intencionalmente.

Na computação tradicional, redes tolerantes a falhas simples (*Crash Fault Tolerance - CFT*, como Raft ou Paxos) assumem que os nós não são maliciosos. A tecnologia **Blockchain** (Nakamoto, 2008) inaugura a **Tolerância a Falhas Bizantinas (BFT)** aberta e em larga escala através da junção de:
1. **Criptografia Assimétrica** (identificação inequívoca de agentes).
2. **Funções Hash Criptográficas** (imutabilidade de registros).
3. **Mecanismos Econômico-Computacionais de Consenso** (custo de trabalho para blindar a verdade histórica).


---
# 2. Pilares Criptográficos: Chaves Assimétricas e Assinatura Digital

Em redes Blockchain, não existem contas com login e senha tradicionais mantidas em tabelas `users` de um servidor. Em vez disso, a autorização e a identidade operam exclusivamente sob o padrão de **Criptografia de Chave Pública**:

$$\text{Par Criptográfico} = \langle \text{Chave Privada } (sk), \text{Chave Pública } (pk) \rangle$$

* **Chave Privada (*Secret Key - sk*):** Um número aleatório de 256 bits mantido sob sigilo absoluto pelo proprietário. É o segredo criptográfico utilizado para **assinar transações** e autorizar a movimentação de valor ou execução de código.
* **Chave Pública (*Public Key - pk*):** Derivada matematicamente da chave privada através de multiplicação em Curvas Elípticas (algoritmo **ECDSA**, na curva padrão **secp256k1**). Pode ser amplamente divulgada na rede e dá origem ao **endereço da carteira** (*Wallet Address*).
* **Propriedade Unidirecional da Curva Elíptica:** Obter $pk$ a partir de $sk$ é trivial em frações de milissegundo:
  $$pk = sk \times G$$
  Entretanto, calcular $sk$ a partir de $pk$ (o Problema do Logaritmo Discreto em Curvas Elípticas) exigiria eras computacionais inatingíveis mesmo para supercomputadores modernos.

### As Quatro Garantias da Assinatura Digital:
1. **Autenticidade:** Comprova que a transação emanou legitimamente do detentor da chave privada correspondente.
2. **Autorização:** Apenas quem possui a posse da chave privada pode assinar uma transação válida.
3. **Integridade:** Qualquer alteração em um único centavo da transação assinada invalida matematicamente a assinatura.
4. **Não-Repúdio:** O remetente não pode negar a autoria de uma transação validada pela sua chave pública.


In [ ]:
# Demonstração Prática: Criptografia Assimétrica e Assinatura de Transações
# Simulando a geração de chaves e a verificação matemática de não-repúdio
import hashlib
import secrets

class CarteiraSimulada:
    def __init__(self, nome):
        self.nome = nome
        # Gerando uma chave privada aleatória de 256 bits (simulando secp256k1)
        self._chave_privada = secrets.token_hex(32)
        # Derivando a chave pública (hash representativo da chave privada)
        self.chave_publica = hashlib.sha256(self._chave_privada.encode()).hexdigest()[:40]
        
    def assinar_transacao(self, mensagem_transacao: str) -> str:
        # A assinatura digital amarra a transação à chave privada do emissor
        payload = f"{mensagem_transacao}::{self._chave_privada}"
        assinatura = hashlib.sha256(payload.encode()).hexdigest()
        return assinatura

def verificar_assinatura(chave_publica, transacao, assinatura, chave_privada_referencia) -> bool:
    # A rede verifica a integridade sem necessitar expor a chave privada original
    esperada = hashlib.sha256(f"{transacao}::{chave_privada_referencia}".encode()).hexdigest()
    return assinatura == esperada

# Testando a criação de uma transação corporativa
alice = CarteiraSimulada("Filial_Sao_Paulo")
print(f"🏛️ Carteira: {alice.nome}")
print(f"🔑 Chave Pública (Endereço Público): 0x{alice.chave_publica}")
print(f"🔒 Chave Privada (Segredo Absoluto): 0x{alice._chave_privada[:8]}...[PROTEGIDA]")

transacao_original = "TRANSFERIR 50000 UNIDADES LOTE_FABRICA_01 PARA MATRIZ"
assinatura = alice.assinar_transacao(transacao_original)
print(f"\n📝 Transação: '{transacao_original}'")
print(f"🔏 Assinatura Digital Gerada: {assinatura}")

# Verificando a autenticidade
valida = verificar_assinatura(alice.chave_publica, transacao_original, assinatura, alice._chave_privada)
print(f"✅ Assinatura Válida na Rede? {valida}")

# Tentativa de Adulteração Maliciosa
transacao_adulterada = "TRANSFERIR 99999 UNIDADES LOTE_FABRICA_01 PARA HACKER"
fraude_valida = verificar_assinatura(alice.chave_publica, transacao_adulterada, assinatura, alice._chave_privada)
print(f"❌ Tentativa de Fraude Validada? {fraude_valida} (Transação rejeitada imediatamente!)")


---
# 3. Funções Hash Criptográficas (SHA-256)

A função hash criptográfica é o bloco de construção mais elementar de uma cadeia de blocos. Trata-se de um algoritmo matemático que aceita uma entrada de dados de **comprimento arbitrário** (um caractere, um contrato de 500 páginas ou um arquivo de 10 Gigabytes) e a transforma em uma cadeia de bits de **comprimento fixo**:

$$H: \{0, 1\}^* \to \{0, 1\}^{256}$$

No padrão **SHA-256** (*Secure Hash Algorithm* de 256 bits, projetado pela NSA e adotado no protocolo Bitcoin):
* A saída tem invariavelmente **256 bits** (32 bytes).
* É expressa usualmente por **64 caracteres hexadecimais** (`[0-9a-f]`).
* O espaço total de combinações possíveis é $2^{256} \approx 1.1579 \times 10^{77}$ (uma quantidade comparável ao número estimado de átomos no universo observável).

```mermaid
flowchart LR
    subgraph Entrada Arbitraria
        A["Texto: 'Olá Mundo'"]
        B["Arquivo: Balanco_2026.pdf (15 MB)"]
        C["Transacao: Alice envia 10 BTC para Bob"]
    end

    H["Função Criptográfica SHA-256"]

    subgraph Hash Saida Fixa 64 Hex
        D["a591a6d40bf420404a011733cfb7b190d62c65bf0bcda32b57b277d9ad9f146e"]
        E["8b88cdb978a05f114c9794e7724a873138b72443657b98d289aa81f21136b6f0"]
        F["e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855"]
    end

    A --> H --> D
    B --> H --> E
    C --> H --> F
```

---

## 3.1 As Cinco Propriedades Canônicas de um Hash Criptográfico

Para que uma função matemática seja qualificada como hash criptográfico seguro para Sistemas de Informação distribuídos, ela deve satisfazer rigorosamente cinco propriedades:

1. **Determinismo Estrito:** Uma mesma entrada $x$ produzirá invariavelmente o mesmíssimo valor hash $H(x)$, independentemente do hardware, sistema operacional ou número de execuções.
2. **Eficiência Computacional (*High Speed*):** O cálculo de $H(x)$ para qualquer entrada deve ser realizado de forma quase instantânea (poucos microssegundos).
3. **Resistência a Pré-Imagem (*One-Way Function*):** Dado um valor hash $y$, deve ser computacionalmente impossível determinar a entrada original $x$ de modo que $H(x) = y$. Não existe algoritmo de "desfazer hash"; a única forma seria por força bruta (tentar todas as combinações aleatórias).
4. **Resistência a Segunda Pré-Imagem e a Colisões:** Deve ser computacionalmente inviável encontrar duas entradas distintas $x_1 \neq x_2$ tais que $H(x_1) = H(x_2)$.
5. **Efeito Avalanche (*Avalanche Effect*):** A alteração de um único bit no dado de entrada (trocar um ponto final por uma vírgula ou uma letra minúscula por maiúscula) acarreta uma modificação caótica e imprevisível em cerca de **50% dos bits da saída**. Não existe correlação linear entre a variação da entrada e a saída.


In [ ]:
# Demonstração do Efeito Avalanche no Algoritmo SHA-256
import hashlib

def sha256_hex(texto: str) -> str:
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()

def sha256_binario(texto: str) -> str:
    # Converte o hash hexadecimal em uma cadeia pura de 256 bits (0s e 1s)
    h_hex = sha256_hex(texto)
    return bin(int(h_hex, 16))[2:].zfill(256)

# Comparando duas entradas que diferem por apenas 1 caractere (ponto final)
entrada_1 = "Transferir R$ 10.000,00 da conta 123 para 456"
entrada_2 = "Transferir R$ 10.000,00 da conta 123 para 456."

hash_1_hex = sha256_hex(entrada_1)
hash_2_hex = sha256_hex(entrada_2)

bin_1 = sha256_binario(entrada_1)
bin_2 = sha256_binario(entrada_2)

# Calculando a Distância de Hamming (quantos bits mudaram entre as saídas)
bits_alterados = sum(b1 != b2 for b1, b2 in zip(bin_1, bin_2))
porcentagem_mudanca = (bits_alterados / 256) * 100

print("=" * 80)
print(f"Entrada 1: '{entrada_1}'")
print(f"Hash 1 (HEX): {hash_1_hex}")
print("-" * 80)
print(f"Entrada 2: '{entrada_2}'")
print(f"Hash 2 (HEX): {hash_2_hex}")
print("=" * 80)
print(f"📊 Total de Bits Analisados: 256 bits")
print(f"⚡ Bits que mudaram de estado (0 <-> 1): {bits_alterados} bits")
print(f"💥 Taxa de Mudança (Efeito Avalanche): {porcentagem_mudanca:.2f}% dos bits alterados!")
print("=" * 80)


---
# 4. Estruturas de Dados em Árvore: A Árvore de Merkle (*Merkle Tree*)

Em um ambiente corporativo ou financeiro de grande escala, um bloco pode conter centenas ou milhares de transações simultâneas. Se fosse necessário verificar individualmente cada registro sequencialmente, a largura de banda e a memória exigidas para auditoria tornariam a rede inviável para clientes leves (smartphones, dispositivos IoT urbanos e servidores de borda).

A solução introduzida por Ralph Merkle (1979) é a **Árvore de Merkle**: uma estrutura hierárquica em árvore binária que sintetiza todas as transações em um único hash mestre de 32 bytes denominado **Raiz de Merkle (*Merkle Root*)**.

![Figura - Arquitetura de Bloco e Árvore de Merkle](images/blockchain_merkle_diagram.jpg)
*Figura 1: Arquitetura de Bloco contendo o Block Header (Previous Hash, Timestamp, Nonce, Merkle Root) conectado à Árvore de Merkle de transações.*

---

## 4.1 Mecanismo de Construção e Propriedades
1. Cada transação bruta $Tx_i$ é individualmente resumida pelo seu hash: $H_i = \text{SHA256}(Tx_i)$.
2. Os hashes adjacentes são concatenados em pares e novamente hasheados:
   $$H_{12} = \text{SHA256}(H_1 \mathbin{\Vert} H_2) \quad \text{e} \quad H_{34} = \text{SHA256}(H_3 \mathbin{\Vert} H_4)$$
3. O processo repete-se recursivamente subindo pelos níveis da árvore até restar um único vértice: a **Merkle Root**:
   $$\text{Merkle Root} = \text{SHA256}(H_{12} \mathbin{\Vert} H_{34})$$

### Vantagem Computacional: Verificação Logarítmica $\mathcal{O}(\log_2 N)$
Para provar que uma transação específica $Tx_3$ está incluída em um bloco com $N = 1.000.000$ transações (Prova de Merkle / *Merkle Proof*), o validador **não precisa baixar o bloco inteiro**. Ele precisa receber apenas a transação e os hashes irmãos ao longo do caminho até a raiz:

$$\text{Quantidade de Hashes necessários} = \log_2(1.000.000) \approx 20 \text{ hashes}$$

Isso permite que dispositivos de IoT em Cidades Inteligentes e gateways leves validem pagamentos e dados telemétricos instantaneamente via **SPV (*Simplified Payment Verification*)**.


In [ ]:
# Implementação de uma Árvore de Merkle (Merkle Tree) em Python
import hashlib

def sha256(val: str) -> str:
    return hashlib.sha256(val.encode('utf-8')).hexdigest()

class MerkleTree:
    def __init__(self, transacoes: list):
        self.transacoes = transacoes
        # Folhas da árvore: hashes individuais de cada transação
        self.folhas = [sha256(tx) for tx in transacoes]
        self.raiz = self._construir_arvore(self.folhas)
        
    def _construir_arvore(self, nos: list) -> str:
        if len(nos) == 1:
            return nos[0]
        
        proximo_nivel = []
        # Agrupando em pares de dois em dois
        for i in range(0, len(nos), 2):
            no_esquerda = nos[i]
            # Se for ímpar, duplica o último elemento (padrão Bitcoin)
            no_direita = nos[i+1] if i + 1 < len(nos) else no_esquerda
            hash_combinado = sha256(no_esquerda + no_direita)
            proximo_nivel.append(hash_combinado)
            
        return self._construir_arvore(proximo_nivel)

# 4 transações corporativas em um lote de ERP
lote_transacoes = [
    "Tx1: Filial SP envia R$ 50.000 para Matriz",
    "Tx2: Filial RJ envia R$ 30.000 para Matriz",
    "Tx3: Pagamento Fornecedor Alpha R$ 12.500",
    "Tx4: Recolhimento de Tributos R$ 8.200"
]

arvore_original = MerkleTree(lote_transacoes)
print("🌳 ÁRVORE DE MERKLE GERADA:")
for idx, (tx, h) in enumerate(zip(lote_transacoes, arvore_original.folhas)):
    print(f"  Folha {idx+1} [Hash: {h[:16]}...] <- '{tx}'")
print(f"\n👑 MERKLE ROOT DO BLOCO: {arvore_original.raiz}")

# Simulando adulteração em 1 único centavo na transação 3
lote_fraudulento = list(lote_transacoes)
lote_fraudulento[2] = "Tx3: Pagamento Fornecedor Alpha R$ 12.500,01" # 1 centavo a mais
arvore_fraude = MerkleTree(lote_fraudulento)

print("\n🚨 SIMULAÇÃO DE ADULTERAÇÃO NA TX3 (+R$ 0,01):")
print(f"Nova Merkle Root Calculada: {arvore_fraude.raiz}")
print(f"As raízes coincidem? {arvore_original.raiz == arvore_fraude.raiz} (Fraude flagrada na Raiz!)")


---
# 5. Anatomia do Bloco, Mineração e Segurança dos Ativos

Um bloco em uma rede Blockchain é uma estrutura de dados dividida em duas regiões interdependentes:

```
+-------------------------------------------------------------------------+
|                              BLOCK HEADER                               |
|  * Block Height / Index (Número de ordem sequencial do bloco)           |
|  * Previous Hash: Hash SHA-256 do cabeçalho do bloco predecessor B_{i-1} |
|  * Merkle Root: Raiz criptográfica agregando todas as transações do bloco|
|  * Timestamp: Registro de data e hora UTC da geração                    |
|  * Nonce (Number used once): Variável ajustada no processo de mineração |
|  * Difficulty / Target: Quantidade exigida de zeros iniciais no hash    |
+-------------------------------------------------------------------------+
|                               BLOCK BODY                                |
|  * Lista completa das transações assinadas digitalmente (Tx_1 ... Tx_N) |
+-------------------------------------------------------------------------+
```

---

## 5.1 O Encadeamento Criptográfico e a Ruptura em Cascata
O vínculo de imutabilidade é formado porque o hash de cada bloco incorpora o hash de seu bloco precedente:

$$\text{Hash}(B_i) = \text{SHA256}\Big(\text{Index}_i \mathbin{\Vert} \text{Timestamp}_i \mathbin{\Vert} H(B_{i-1}) \mathbin{\Vert} \text{MerkleRoot}_i \mathbin{\Vert} \text{Nonce}_i\Big)$$

* **Bloco Gênesis (*Genesis Block* / Bloco #1):** O bloco número 1 é o marco zero da rede. Como não possui antecessor, seu `Previous Hash` é preenchido convencionalmente com 64 zeros (`0000000000000000000000000000000000000000000000000000000000000000`).
* **Ruptura em Cascata (*Cascading Invalidation*):** Qualquer alteração em um dado no Bloco $B_2$ altera sua Merkle Root e gera um novo Hash para $B_2$. Como o Bloco $B_3$ guardava o hash original de $B_2$, a conexão matemática quebra instantaneamente. Todos os blocos subsequentes ($B_3, B_4, B_5...$) tornam-se inválidos em efeito dominó.

---

## 5.2 O que é REALMENTE Minerar? (Desmistificando o Processo)
Existe um equívoco comum de senso comum de que minerar seria *"resolver equações matemáticas complexas de cálculo"* ou *"procurar moedas digitais escondidas no computador"*. Em termos formais de Engenharia de Software e Sistemas de Informação, **minerar consiste no seguinte fluxo de trabalho computacional:**

```mermaid
flowchart TD
    A["1. Coleta da Mempool: Nós mineradores recebem milhares de transações pendentes transmitidas pela rede"] --> B["2. Validação Criptográfica: O minerador verifica se cada transação possui assinatura digital válida e saldo"]
    B --> C["3. Montagem da Árvore de Merkle: Constrói a árvore e gera a Merkle Root"]
    C --> D["4. Construção do Cabeçalho: Insere Previous Hash, Timestamp e inicializa Nonce = 0"]
    D --> E["5. Prova de Trabalho (PoW): Calcula SHA256(Cabeçalho). O hash inicia com a quantidade exigida de zeros?"]
    E -- "NÃO (Inválido)" --> F["Nonce = Nonce + 1 (Tenta o próximo número por força bruta)"]
    F --> E
    E -- "SIM (Válido!)" --> G["6. Transmissão e Recompensa: Propaga o bloco minerado para a rede P2P e recebe a Coinbase + Taxas!"]
```

### Por que a Mineração é Necessária?
1. **Criação de Escassez Temporal e Custo Econômico:** Como as funções hash são imprevisíveis (efeito avalanche), a única forma de achar um hash válido é testando bilhões de números aleatórios (*Nonce*). Isso exige gasto de energia real (eletricidade e hardware).
2. **Prevenção contra Ataques Sybil e Fraudes:** Ninguém pode criar 1 milhão de blocos falsos por segundo para dominar o sistema, porque cada bloco exige prova de trabalho computacional real.
3. **Incentivo Financeiro:** O minerador que encontra o bloco válido recebe a **Recompensa do Bloco (*Block Reward / Coinbase*)** (novas moedas criadas pela rede) mais as **taxas de transação (*Tx fees*)** pagas pelos usuários.

---

## 5.3 Dúvida Crucial de Sala de Aula:
### *"Se alguém adulterar ou invalidar um bloco anterior, eu perco o meu Bitcoin / meus ativos no bloco seguinte?"*

> ### 🛑 Resposta Direta e Tranquilizadora: **NÃO! VOCÊ NÃO PERDE O SEU BITCOIN!**
>
> Se você possui 1 Bitcoin registrado no Bloco #3, e um invasor mal-intencionado alterar uma transação ocorrida no Bloco #2, **o seu patrimônio e a sua transação continuam 100% seguros e intactos**.

![Figura - Segurança de Ativos e Rejeição de Adulteração](images/seguranca_adulteracao_bitcoin.jpg)
*Figura 2: Por que você NÃO perde seu ativo quando alguém adultera um bloco passado. A adulteração é meramente local (no computador do invasor). A rede descentralizada (Peers honestos) rejeita sumariamente o bloco adulterado, preservando o seu Bitcoin na cadeia canônica.*

### Compreendendo os 4 Motivos Técnicos:

1. **A Adulteração é Puramente Local (No Computador do Invasor):**
   Quando um invasor abre o banco de dados ou o simulador e muda uma transação no Bloco #2, **ele só alterou o arquivo no disco rígido DELE**. Ele não tem a menor capacidade de alterar remotamente os discos rígidos dos outros dezenas de milhares de nós e validadores espalhados pelo mundo.

2. **A Rede P2P Global Rejeita a Fraude Instantaneamente:**
   Quando o invasor tenta transmitir o seu bloco adulterado para a rede mundial, os nós honestos (Peer A, Peer C, corretoras, mineradores) recalculam o hash do bloco recebido. Constatando que o hash não confere com a cadeia válida ou que quebrou a regra de dificuldade, **os nós honestos simplesmente rejeitam e descartam o bloco do invasor**.

3. **O Seu Bitcoin Está Gravado na Cadeia Canônica Oficial:**
   Os nós da rede mundial continuam reconhecendo como verdade a **Cadeia Canônica Honesta** (aquela que possui a maior prova de trabalho acumulada e dados consistentes). Como o seu Bitcoin reside no Bloco #3 da cadeia canônica, o seu saldo permanece inalterado.

4. **E se houver uma bifurcação temporária legítima (*reorg*)?**
   Mesmo no caso de dois mineradores honestos encontrarem blocos válidos no mesmo segundo (criando uma breve bifurcação), a rede aguarda o bloco seguinte para decidir qual ramo crescerá mais rápido. Caso o bloco onde estava sua transação seja descartado (bloco órfão), **sua transação simplesmente retorna para a Mempool e é incluída no bloco imediatamente posterior**. A sua posse nunca é destruída nem perdida, pois a assinatura digital pertence exclusivamente à sua chave privada!


In [ ]:
# Implementação de uma Cadeia de Blocos Funcional (Mini-Blockchain) em Python
import hashlib
import time

class Bloco:
    def __init__(self, indice: int, previous_hash: str, transacoes: str, nonce: int = 0):
        self.indice = indice
        self.timestamp = time.time()
        self.previous_hash = previous_hash
        self.transacoes = transacoes
        self.nonce = nonce
        self.hash = self.calcular_hash()
        
    def calcular_hash(self) -> str:
        conteudo = f"{self.indice}{self.timestamp}{self.previous_hash}{self.transacoes}{self.nonce}"
        return hashlib.sha256(conteudo.encode('utf-8')).hexdigest()
    
    def minerar(self, dificuldade: int):
        # Prova de Trabalho simplificada: encontrar hash que comece com 'dificuldade' zeros
        alvo = "0" * dificuldade
        while not self.hash.startswith(alvo):
            self.nonce += 1
            self.hash = self.calcular_hash()
        print(f"⛏️ Bloco #{self.indice} Minerado! Nonce: {self.nonce} | Hash: {self.hash[:24]}...")

class MiniBlockchain:
    def __init__(self, dificuldade: int = 3):
        self.cadeia = [self._criar_bloco_genesis()]
        self.dificuldade = dificuldade
        
    def _criar_bloco_genesis(self) -> Bloco:
        bloco_g = Bloco(0, "0" * 64, "BLOCO GENESIS: INAUGURACAO DO SISTEMA", nonce=0)
        bloco_g.minerar(1)
        return bloco_g
    
    def adicionar_bloco(self, transacoes: str):
        ultimo_bloco = self.cadeia[-1]
        novo_bloco = Bloco(indice=len(self.cadeia), previous_hash=ultimo_bloco.hash, transacoes=transacoes)
        novo_bloco.minerar(self.dificuldade)
        self.cadeia.append(novo_bloco)
        
    def verificar_integridade(self) -> bool:
        for i in range(1, len(self.cadeia)):
            atual = self.cadeia[i]
            anterior = self.cadeia[i - 1]
            
            # 1. O hash do bloco atual recalcula perfeitamente?
            if atual.hash != atual.calcular_hash():
                print(f"🚨 ERRO: Hash do Bloco #{atual.indice} foi recalculado e não confere!")
                return False
            # 2. O ponteiro Previous Hash confere com o hash do anterior?
            if atual.previous_hash != anterior.hash:
                print(f"🚨 ERRO: Quebra de Encadeamento no Bloco #{atual.indice}! PrevHash incorreto!")
                return False
        return True

# Instanciando e minerando blocos
bc = MiniBlockchain(dificuldade=2)
bc.adicionar_bloco("Lote A: 100 sensores IoT cadastrados no Gateway Urbano")
bc.adicionar_bloco("Lote B: Registro de Nota Fiscal Eletrônica #45902")
bc.adicionar_bloco("Lote C: Baixa contábil de R$ 15.000 em estoque")

print(f"\n🛡️ A Cadeia de Blocos é íntegra? {bc.verificar_integridade()}")

# Injetando fraude no Bloco 1 (adulterando os dados após a mineração)
print("\n💣 INJETANDO ADULTERAÇÃO NO BLOCO #1...")
bc.cadeia[1].transacoes = "Lote A: 9999 SENSORES FALSOS INJETADOS POR HACKER"
# O invasor tenta disfarçar a fraude recalculando o hash do próprio bloco
bc.cadeia[1].hash = bc.cadeia[1].calcular_hash()

print(f"🛡️ A Cadeia de Blocos ainda é válida após a invasão? {bc.verificar_integridade()}")


---
# Roteiro de Laboratório Prático Dirigido (~40 minutos)
## Simulador Interativo Oficial: [https://blockchaindemo.io/](https://blockchaindemo.io/)

A partir deste ponto da aula, os estudantes deverão acessar individualmente ou em duplas o simulador visual **[https://blockchaindemo.io/](https://blockchaindemo.io/)** para conduzir o protocolo experimental de validação e quebra em cascata.

O simulador é estruturado em etapas sequenciais acessíveis pelo menu de navegação:
1. **Hash**
2. **Block**
3. **Blockchain**
4. **Distributed**
5. **Tokens** / **Coinbase**

---

### Atividade de Laboratório 1: O Comportamento do Hash Criptográfico
* **Objetivo Prático:** Observar o determinismo, a velocidade de cálculo e o Efeito Avalanche em tempo real.

#### Passo a Passo:
1. Acesse o site [https://blockchaindemo.io/](https://blockchaindemo.io/) e clique na primeira seção: **Hash**.
2. No campo **Data**, digite seu nome completo exatamente como registrado na instituição.
3. Observe o hash hexadecimal gerado instantaneamente no rodapé da caixa.
4. Agora, adicione um único ponto final (`.`) ou um espaço ao final do seu nome.
5. Observe como todos os caracteres do hash mudaram de forma radical e imprevisível.
6. Apague o caractere adicionado: constate que o hash original retorna com determinismo absoluto.

> 📝 **Anotação de Laboratório 1:**
> * Qual foi o hash gerado para o seu nome?
> * Anote os 8 primeiros caracteres antes e depois de inserir o ponto final:
>   * *Sem ponto:* `0x...`
>   * *Com ponto:* `0x...`


---
### Atividade de Laboratório 2: A Anatomia do Bloco e a Mineração (*Nonce*)
* **Objetivo Prático:** Compreender a mecânica do *Nonce*, a exigência do *Target* (dificuldade) e o custo de mineração.

#### Passo a Passo:
1. Navegue até a aba **Block** no simulador.
2. Identifique os quatro campos principais exibidos no card do bloco:
   * `Block:` Número de ordem do bloco (ex: `1`).
   * `Nonce:` Número inteiro arbitrário inicial.
   * `Data:` Área de texto contendo os registros do bloco.
   * `Hash:` Hash resultante de todos os campos combinados.
3. Observe a **cor de fundo do bloco**:
   * **Verde:** O hash gerado satisfaz a regra de dificuldade da rede (inicia com uma sequência de zeros predefinida, ex: `0000...`).
   * **Vermelho:** O bloco é inválido perante as regras de consenso da rede.
4. No campo **Data**, escreva: `Auditoria de Almoxarifado - Lote #78901 Concluída`.
5. **O que aconteceu imediatamente com o card do bloco?** Ele mudou para a cor **vermelha**!
6. Clique no botão azul **Mine**:
   * Observe o número do `Nonce` incrementando em alta velocidade no navegador.
   * O navegador está executando um laço `while (hash != 0000...) Nonce++`.
   * Quando o hash gerado finalmente inicia com os zeros exigidos, a busca cessa e o bloco volta a ficar **verde**.

> 📝 **Anotação de Laboratório 2:**
> * Qual era o valor do Nonce antes de minerar?
> * Qual foi o valor final do Nonce encontrado pelo botão *Mine* para validar seus dados?
> * Por que um computador não pode simplesmente "deduzir" o Nonce ideal matematicamente sem precisar testar um a um por força bruta?


---
### Atividade de Laboratório 3: A Cadeia de Blocos (*Blockchain*) e a Ruptura em Cascata
* **Objetivo Prático:** Vivenciar a resistência matemática contra fraudes no histórico contábil.

#### Passo a Passo:
1. Navegue até a aba **Blockchain** no simulador.
2. Você verá uma sequência encadeada de 5 blocos consecutivos (`Block #1` até `Block #5`).
3. Analise o **Block #1 (Bloco Gênesis)**:
   * Observe o campo `Prev:` preenchido com zeros (`000000000000...`).
4. Analise o **Block #2**:
   * Observe que o campo `Prev:` do Bloco #2 é **idêntico caractere por caractere** ao campo `Hash:` do Bloco #1.
5. **O Experimento do Invasor (Ataque de Adulteração):**
   * Vá até o **Block #2**.
   * No campo `Data`, insira uma alteração fraudulenta: `Transação Falsa: Desviar R$ 1.000.000,00 para Conta X`.
6. **Observe o que ocorre instantaneamente:**
   * O Bloco #2 torna-se vermelho (inválido).
   * **Os Blocos #3, #4 e #5 também se tornam vermelhos simultaneamente em efeito dominó!**
7. **A Tentativa de Fraude do Invasor:**
   * Clique no botão **Mine** do Bloco #2 para "consertar" o bloco adulterado.
   * O Bloco #2 volta a ficar verde.
   * **Contudo, os Blocos #3, #4 e #5 continuam vermelhos e inválidos!**
   * Para que a cadeia voltasse a ser aceita, o invasor seria forçado a re-minerar o Bloco #3, depois o Bloco #4 e depois o Bloco #5 em sequência, antes que o resto da rede mundial gerasse novos blocos legítimos.

> 📝 **Anotação de Laboratório 3:**
> * Explique por que a mineração bem-sucedida do Bloco #2 NÃO foi suficiente para validar o Bloco #3.
> * Qual é a implicação prática dessa dinâmica para a segurança de dados em Sistemas de Informação empresariais?


---
### Atividade de Laboratório 4: A Rede Peer-to-Peer Distribuída e a Regra do Consenso
* **Objetivo Prático:** Vivenciar na prática por que você **NÃO perde seu Bitcoin** quando um nó vizinho sofre uma adulteração.

#### Passo a Passo:
1. Navegue até a aba **Distributed** no simulador [https://blockchaindemo.io/](https://blockchaindemo.io/).
2. Observe que a interface apresenta três nós independentes conectados em rede P2P:
   * `Peer A`
   * `Peer B`
   * `Peer C`
3. Verifique o **Hash final do Bloco #5** em todos os três peers: constate que todos compartilham rigorosamente o mesmo hash final na ponta da cadeia. Existe **consenso absoluto de rede**.
4. **Cenário de Teste: O Ataque no Peer B e a Segurança do seu Ativo no Peer A:**
   * Suponha que o **seu Bitcoin** está registrado com segurança no **Bloco #3 do Peer A**.
   * Agora, um funcionário mal-intencionado ou invasor no **Peer B** adultera os dados do **Bloco #2** para tentar forjar um saldo falso: `Desvio Ilegal de Fundos`.
   * O Bloco #2 do Peer B torna-se vermelho instantaneamente. O invasor clica em *Mine* no Bloco #2, Bloco #3, Bloco #4 e Bloco #5 na máquina dele, até deixar a cadeia dele visualmente verde no navegador dele.
5. **O Julgamento da Rede Distribuída:**
   * Olhe para o `Peer A` (onde está o seu Bitcoin) e para o `Peer C`:
   * **O seu Bitcoin no Peer A foi afetado em alguma coisa?** **NÃO!** O hash do Bloco #3 e do Bloco #5 no Peer A continua absolutamente idêntico ao do Peer C.
   * O Peer B agora tem um hash final completamente divergente da maioria da rede.
   * Pela **Regra do Consenso da Maioria e da Cadeia Válida Mais Longa (*Longest Chain Rule*)**, os nós honestos da rede (Peer A e Peer C) ignoram a cadeia proposta pelo Peer B. A fraude morreu isolada na máquina do invasor!

> 📝 **Anotação de Laboratório 4:**
> * O que aconteceu com a cadeia do Peer B em relação aos Peers A e C?
> * Por que o usuário que possui fundos no Peer A ou Peer C não sofreu nenhuma perda patrimonial com a adulteração ocorrida no Peer B?
> * O que seria necessário para que a versão fraudulenta do Peer B fosse aceita pela rede inteira? (Conceito de Ataque dos 51%).


---
# 3. Protocolo de Avaliação Formativa da Aula 01
### Resolução Individual ou em Duplas (Entrega Obrigatória ao Final da Aula)

Responda fundamentadamente às cinco questões dissertativas abaixo com base na teoria apresentada e nas evidências experimentais coletadas no simulador [blockchaindemo.io](https://blockchaindemo.io/).

---

### Questão 1 (Mecanismos Criptográficos de Integridade):
*Diferencie conceitualmente uma função hash comum (como as usadas em tabelas hash em memória) de uma função hash criptográfica (como o SHA-256). Em seguida, explique por que o **Efeito Avalanche** inviabiliza que um invasor deduza gradualmente o dado de entrada através de tentativas aproximadas.*

**Espaço para Resposta do Estudante:**
```markdown
[DIGITE SUA RESPOSTA AQUI]



```

---

### Questão 2 (Arquitetura e Escalabilidade com Árvores de Merkle):
*Em um Sistema de Informação bancário ou corporativo que processe 2.000 transações por segundo, qual seria o gargalo operacional se o cabeçalho de cada bloco contivesse a listagem de todas as transações em texto plano? Como a **Merkle Root** e as **Merkle Proofs** resolvem esse problema para dispositivos móveis e sensores de Cidades Inteligentes?*

**Espaço para Resposta do Estudante:**
```markdown
[DIGITE SUA RESPOSTA AQUI]



```

---

### Questão 3 (O que é Minerar):
*Explique tecnicamente o que um computador está fazendo quando executa o processo de 'mineração' de um bloco em uma rede com Prova de Trabalho (PoW). Qual é a função do campo **Nonce** e por que esse mecanismo consome tanta eletricidade?*

**Espaço para Resposta do Estudante:**
```markdown
[DIGITE SUA RESPOSTA AQUI]



```

---

### Questão 4 (Ruptura em Cascata no Simulador):
*Com base na experiência realizada na aba **Blockchain** do simulador, descreva a sequência de eventos que ocorrem a partir do momento em que um único caractere é adulterado no Bloco #2 até a invalidação do Bloco #5. Por que re-minerar apenas o Bloco #2 não conserta o restante da cadeia?*

**Espaço para Resposta do Estudante:**
```markdown
[DIGITE SUA RESPOSTA AQUI]



```

---

### Questão 5 (Segurança Patrimonial e Imutabilidade Coletiva):
*Um estudante novato pergunta aflito em sala: 'Professor, se eu tenho 1 Bitcoin registrado no Bloco #4, e alguém adulterar o Bloco #3, eu perco o meu Bitcoin?'. Responda a esse estudante com base no experimento da aba **Distributed**, explicando por que alterações locais não corrompem a cadeia canônica e como o consenso P2P protege a posse dos usuários honestos.*

**Espaço para Resposta do Estudante:**
```markdown
[DIGITE SUA RESPOSTA AQUI]



```

---

### Rubrica de Avaliação Formativa:
| Critério de Desempenho | Insuficiente (0-4) | Regular (5-7) | Excelente (8-10) |
| :--- | :--- | :--- | :--- |
| **Rigor Teórico e Vocabulário** | Confunde hash com criptografia reversível; não cita propriedades do SHA-256 nem compreende mineração. | Descreve o hash mas comete imprecisões sobre Merkle Tree, chaves ou mineração. | Domina formalmente propriedades (determinismo, efeito avalanche, $O(\log n)$, Nonce, BFT). |
| **Rastreabilidade Experimental** | Não executou os testes no simulador; respostas sem dados empíricos. | Executou os testes mas não articulou a evidência empírica aos conceitos arquiteturais. | Articula com clareza o comportamento observado no simulador (cor vermelha, Nonce, divergência entre Peers). |
| **Compreensão de Consenso e Posse** | Acredita que adulteração local apaga saldos na rede inteira. | Compreende que a rede rejeita blocos, mas titubeia sobre a persistência dos saldos. | Explica com perfeição por que a cadeia canônica blinda os saldos e neutraliza adulterações locais. |
| **Aplicação em Sistemas de Informação** | Não consegue transpor os conceitos para aplicações empresariais reais. | Cita apenas casos de criptomoedas financeiras sem abstrair para SI corporativos. | Relaciona DLT com auditoria, prevenção de SPOF, rastreabilidade e integridade em SI. |


---
# 4. Bibliografia e Próximos Passos

### Gancho para a Próxima Aula (Aula 02):
Na **Aula 02**, aprofundaremos como a rede decide quem tem o direito de propor o próximo bloco através dos **Mecanismos de Consenso Avançados** (*Proof of Work* vs. *Proof of Stake* na Ethereum 2.0 vs. *Proof of Authority* em redes privadas Hyperledger). Estudaremos como a **Ethereum Virtual Machine (EVM)** expandiu o Blockchain de uma planilha financeira para um computador mundial Turing-completo e a função econômica do **Gas**.

### Bibliografia Canônica Desta Aula:
1. **NAKAMOTO, Satoshi.** *Bitcoin: A Peer-to-Peer Electronic Cash System*. 2008. Disponível em: <https://bitcoin.org/bitcoin.pdf>.
2. **ANTONOPOULOS, Andreas M.** *Mastering Bitcoin: Programming the Open Blockchain*. 2. ed. Sebastopol: O'Reilly Media, 2017.
3. **MERKLE, Ralph C.** *A Certified Digital Signature*. In: Advances in Cryptology — CRYPTO '79. Springer, 1979. p. 218-238.
4. **LAMPORT, Leslie; SHOSTAK, Robert; PEASE, Marshall.** *The Byzantine Generals Problem*. ACM Transactions on Programming Languages and Systems, v. 4, n. 3, p. 382-401, 1982.
5. **NARAYANAN, Arvind et al.** *Bitcoin and Cryptocurrency Technologies: A Comprehensive Introduction*. Princeton: Princeton University Press, 2016.
6. **BROWNWORTH, Anders; HAN, Sean.** *Blockchain Demo: A visual demo of blockchain technology*. Disponível em: <https://blockchaindemo.io/>.
